# OULAD exploration and data preparation

This notebook is the walkthrough for `ml/src/explore_oulad.py`. It calls the same reproducible Pandas analysis rather than copying results into notebook cells. No learner-level rows are displayed or exported.

## 1. Dataset discovery and loading

The explorer requires all seven OULAD source files. Put them in `ml/data/raw/` or set `DATA_DIR` below to their existing local directory. The script fails with a clear message if any required file is missing.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'ml' / 'src' / 'explore_oulad.py').exists():
    REPO_ROOT = Path.cwd().parent
SCRIPT = REPO_ROOT / 'ml' / 'src' / 'explore_oulad.py'
DATA_DIR = REPO_ROOT / 'ml' / 'data' / 'raw'  # e.g. Path('D:/my files/anonymisedData')
REPORTS_DIR = REPO_ROOT / 'ml' / 'reports'
SAMPLE_SIZE = 6000  # Configure between 5,000 and 7,000 for development
SEED = 42

result = subprocess.run([sys.executable, str(SCRIPT), '--data-dir', str(DATA_DIR),
                         '--reports-dir', str(REPORTS_DIR), '--sample-size', str(SAMPLE_SIZE),
                         '--seed', str(SEED)], text=True, capture_output=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError('OULAD exploration failed; see the error above.')

## 2. Dimensions and schema

`csv_structure_summary.csv` lists actual dimensions, observed column names and Pandas dtypes, plus candidate keys for each loaded file.

## 3. Missing values and duplicate checks

`missing_values_summary.csv` gives per-column counts and percentages. `data_quality_summary.csv` reports exact duplicate rows and duplicate/null counts for each expected candidate key.

## 4. Key uniqueness and relationships

The proposed unit is the composite learner-course-presentation key: `id_student`, `code_module`, `code_presentation`. Inspect aggregate key checks before joining. Student assessment records connect to assessment metadata through `id_assessment`; VLE events connect to VLE resource metadata through `id_site`, module, and presentation. Registration joins on the learner-course-presentation key.

## 5. `final_result` and proposed target

The report records actual outcome counts. Initial proposal: `Withdrawn` = 1; `Pass`, `Fail`, and `Distinction` = 0; missing/unexpected outcomes remain unknown and are excluded. Team confirmation is required. `final_result` constructs the target and must never be an input feature.

## 6. Assessment, VLE, registration and course features

The generated feature dictionary records proposed calculations, source tables, join keys, cutoff availability, missingness and risks. Assessment due/submission/score timing needs validation. Candidate VLE summaries include clicks, active days, last activity and trends, calculated only after restricting events to the cutoff. Registration timing needs relative-day validation; unregistration is excluded because it may expose future outcomes.

## 7. Merge plan and one-row-per-unit design

Use `studentInfo` as the base. Join course and registration metadata after validating key cardinality. Join and aggregate assessment events separately from VLE events, then attach each aggregate to base on the composite key. Never join the two event-level tables directly. Check row counts after each join and enforce one row per composite key.

## 8. Cutoff and leakage risks

Day 30 after presentation start is a provisional assumption for team discussion, not an approved definition. Dates are relative days, not calendar dates. Apply `date <= 30` before aggregating event features. Exclude `final_result` and post-cutoff behavior from predictors. Do not treat missing scores as proof of missed assessments or missing unregistration as completion. Review sensitive learner attributes for privacy and fairness.

## 9. Findings, assumptions and unresolved decisions

Read `reports/target_and_merge_plan.md` for counts derived from the current files, sample size/class mix, merge sequence, limitations and decisions the team must confirm. Sampling is uniform without replacement from unique learners, retaining all their course-presentation rows, with a fixed seed. Event aggregation must happen after filtering to sampled keys and applying the prediction cutoff. Phase 1 defines the training sample but does not train a model.